# Image Captioning — encoder/decoder architecture comparison

This notebook runs an **architecture search** for image captioning: instead of tuning numeric hyperparameters, we compare combinations of an image **encoder** with a text **decoder** and score each by validation BLEU-4 (exactly like the earlier grid search, but over *architectures only*).

Everything is wired through one **unified framework** so any encoder can feed any decoder:

- **Encoders** (image → sequence of feature vectors):
  - `cnn` — frozen ResNet-50 (the backbone from the earlier CNN+RNN notebook), conv feature map → 49 region tokens.
  - `vit` — ViT (`google/vit-base-patch16-224-in21k`).
  - `clip` — CLIP vision tower (`openai/clip-vit-base-patch32`).
- **Decoders** (features → caption):
  - `gru` — word-level GRU (the decoder from the earlier notebook): mean-pools the encoder sequence into a first input token.
  - `gpt2` — GPT-2 with cross-attention over the encoder sequence (subword tokenizer).

Because the encoder and decoder are decoupled, we can mix them freely — e.g. **CNN + GPT-2** or **ViT + GRU** — not just the pairs Hugging Face's `VisionEncoderDecoderModel` allows. The encoder is frozen (feature extractor) so the comparison is fast and fair; the decoder (and any projection / cross-attention) is trained.

Defaults use `val2017` to keep compute reasonable. Built for Colab (also runs on a local Jupyter).</cell id="cell-0">

## 1. Install dependencies and imports

In [ ]:
# Install once if needed (transformers powers the ViT/CLIP encoders + GPT-2 decoder)
!pip -q install transformers pycocotools nltk

import os, json, random, time, copy
from dataclasses import dataclass, asdict, field
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T
from torchvision import models

from transformers import (
    AutoConfig, AutoModel, AutoModelForCausalLM, AutoTokenizer,
    AutoImageProcessor, CLIPVisionModel,
)

import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
from nltk.tokenize import word_tokenize
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Using device:", device)

## 2. Project settings

In [ ]:
# Portable paths: works on Colab AND a local Jupyter notebook.
try:
    import google.colab  # noqa: F401
    BASE_DIR = "/content"
except ImportError:
    BASE_DIR = os.path.abspath(".")

DATA_DIR = os.path.join(BASE_DIR, "data", "coco")
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs_transformer")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

COCO_SPLIT = "val"      # "val" for a smaller project run; "train" for full COCO if downloaded
FAST_DEV_RUN = False    # True only while debugging

# Training budget per architecture (subset training, like the earlier grid search)
EPOCHS = 1 if FAST_DEV_RUN else 3
MAX_TRAIN_BATCHES = 3 if FAST_DEV_RUN else 150   # batches/epoch cap (None = full train set)
BATCH_SIZE = 16
EVAL_NUM_BATCHES = 3 if FAST_DEV_RUN else 10     # val batches used for BLEU
MAX_TEXT_LEN = 40       # max caption length (tokens) for training labels
MAX_GEN_LEN = 40        # max tokens to generate at inference
NUM_WORKERS = 2
FREEZE_ENCODER = True   # freeze the encoder tower; train decoder + projection/cross-attention

# Learning rate depends on the decoder family: the small word-level GRU likes a
# larger LR; the pretrained GPT-2 needs a gentle fine-tuning LR.
LR_BY_DECODER = {"gru": 1e-3, "gpt2": 5e-5}

# Word-level GRU decoder dims (match the earlier grid-search winner: e256/h512/L2/d0.5).
GRU_EMBED_SIZE = 256
GRU_HIDDEN_SIZE = 512
GRU_NUM_LAYERS = 2
GRU_DROPOUT = 0.5
GRU_FREQ_THRESHOLD = 5

print("Data dir:", DATA_DIR)
print("Output dir:", OUTPUT_DIR)
print("Split:", COCO_SPLIT + "2017")
print("Fast dev run:", FAST_DEV_RUN)

## 3. Download MS-COCO data

Downloaded and extracted with pure Python (`urllib` + `zipfile`), so it runs on Colab or a local Jupyter. The existence checks skip the work if the data is already present. If you already ran the CNN+RNN notebook in the same session, the data is reused.

In [ ]:
import urllib.request, zipfile

ANNOTATIONS_URL = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"
IMAGES_URL = ("http://images.cocodataset.org/zips/val2017.zip" if COCO_SPLIT == "val"
              else "http://images.cocodataset.org/zips/train2017.zip")

ann_dir = os.path.join(DATA_DIR, "annotations")
img_dir = os.path.join(DATA_DIR, f"{COCO_SPLIT}2017")

def _download_and_extract(url, zip_path, extract_to):
    def _progress(block_num, block_size, total_size):
        if total_size > 0:
            pct = min(100, block_num * block_size * 100 / total_size)
            print(f"\r  downloading... {pct:5.1f}%", end="")
    print("Downloading", url)
    urllib.request.urlretrieve(url, zip_path, _progress)
    print("\n  extracting...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_to)
    os.remove(zip_path)
    print("  done.")

if not os.path.exists(ann_dir):
    _download_and_extract(ANNOTATIONS_URL, os.path.join(DATA_DIR, "annotations.zip"), DATA_DIR)
else:
    print("Annotations already extracted.")

if not os.path.exists(img_dir):
    _download_and_extract(IMAGES_URL, os.path.join(DATA_DIR, f"{COCO_SPLIT}2017.zip"), DATA_DIR)
else:
    print(f"{COCO_SPLIT}2017 images already extracted.")

## 4. Load captions and split by image id

Same leakage-safe split as the CNN+RNN notebook: split by **image id** so an image never appears in both train and validation. Text encoding is decoder-specific and happens later: the GRU decoder builds a word-level vocabulary (next section), while the GPT-2 decoder uses its own subword tokenizer.

In [ ]:
annotations_file = os.path.join(DATA_DIR, "annotations", f"captions_{COCO_SPLIT}2017.json")
with open(annotations_file, "r") as f:
    coco_data = json.load(f)

img_id_to_filename = {img["id"]: img["file_name"] for img in coco_data["images"]}
img_id_to_captions = defaultdict(list)
for ann in coco_data["annotations"]:
    img_id_to_captions[ann["image_id"]].append(ann["caption"])

def make_image_id_split(image_ids, train_fraction=0.90, seed=42):
    image_ids = list(image_ids)
    rng = random.Random(seed)
    rng.shuffle(image_ids)
    split_idx = int(train_fraction * len(image_ids))
    return set(image_ids[:split_idx]), set(image_ids[split_idx:])

train_img_ids, val_img_ids = make_image_id_split(img_id_to_filename.keys(), 0.90, SEED)
train_annotations = [a for a in coco_data["annotations"] if a["image_id"] in train_img_ids]
val_annotations = [a for a in coco_data["annotations"] if a["image_id"] in val_img_ids]

assert train_img_ids.isdisjoint(val_img_ids)
print("Train images:", len(train_img_ids), " Val images:", len(val_img_ids))
print("Train captions:", len(train_annotations), " Val captions:", len(val_annotations))

## 5. Dataset

The dataset returns the raw PIL image, the caption string, and the image id. Image preprocessing (encoder-specific) and text encoding (decoder-specific) both happen in a per-architecture `collate` function built by `make_collate(bundle)`.

In [ ]:
class CocoCaptionDataset(Dataset):
    """Returns (PIL image, caption string, image_id)."""
    def __init__(self, img_dir, annotations, img_id_to_filename):
        self.img_dir = img_dir
        self.annotations = list(annotations)
        self.img_id_to_filename = dict(img_id_to_filename)

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        ann = self.annotations[idx]
        img_id = ann["image_id"]
        path = os.path.join(self.img_dir, self.img_id_to_filename[img_id])
        image = Image.open(path).convert("RGB")
        return image, ann["caption"], img_id

train_dataset = CocoCaptionDataset(img_dir, train_annotations, img_id_to_filename)
val_dataset = CocoCaptionDataset(img_dir, val_annotations, img_id_to_filename)
print("Datasets ready:", len(train_dataset), "train rows,", len(val_dataset), "val rows")

## 6. Unified encoder/decoder framework, training, and BLEU evaluation

This is the heart of the notebook. Every model is a `CaptioningModel(encoder, decoder)`:

- **`ImageEncoder(kind, name)`** always returns a sequence of feature vectors `(B, S, D_enc)`:
  - `cnn` → ResNet-50 conv map `(B, 2048, 7, 7)` reshaped to `49` region tokens `(B, 49, 2048)`.
  - `vit` / `clip` → the transformer's `last_hidden_state` `(B, T, 768)`.
- **`GRUDecoder`** mean-pools the encoder sequence into a single vector, projects it to the embedding size, and uses it as the first input token of a word-level GRU (same recipe as the earlier notebook).
- **`GPT2Decoder`** projects the encoder sequence to GPT-2's width and lets GPT-2 **cross-attend** to it. GPT-2's `generate()` doesn't forward `encoder_hidden_states`, so we run a small **manual greedy decode loop** (with KV-caching) that passes the encoder features at every step.

Because both decoders consume the same `(B, S, D_enc)` interface, any encoder pairs with any decoder. The encoder is frozen; a single `build_model(encoder_spec, decoder_kind)` returns the model plus everything the collate/eval code needs.</cell id="cell-11">

In [ ]:
# ===========================================================================
# Unified encoder/decoder framework
# ===========================================================================

# ---- Word-level vocabulary (for the GRU decoder; built from train captions) ----
class Vocabulary:
    def __init__(self, freq_threshold=5):
        self.freq_threshold = freq_threshold
        self.word2idx = {"<pad>": 0, "<start>": 1, "<end>": 2, "<unk>": 3}
        self.idx2word = {0: "<pad>", 1: "<start>", 2: "<end>", 3: "<unk>"}
        self.word_count = Counter()

    def build(self, captions):
        for cap in captions:
            self.word_count.update(word_tokenize(cap.lower()))
        idx = 4
        for w, c in self.word_count.items():
            if c >= self.freq_threshold:
                self.word2idx[w] = idx; self.idx2word[idx] = w; idx += 1

    def numericalize(self, cap):
        ids = [self.word2idx["<start>"]]
        ids += [self.word2idx.get(t, self.word2idx["<unk>"]) for t in word_tokenize(cap.lower())]
        ids.append(self.word2idx["<end>"])
        return ids

    def decode(self, ids):
        words = []
        for i in ids:
            w = self.idx2word.get(int(i), "<unk>")
            if w == "<end>": break
            if w not in ("<start>", "<pad>"): words.append(w)
        return " ".join(words)

    def __len__(self):
        return len(self.word2idx)

rnn_vocab = Vocabulary(freq_threshold=GRU_FREQ_THRESHOLD)
rnn_vocab.build([a["caption"] for a in train_annotations])
print("GRU word-level vocab size:", len(rnn_vocab))

# ---- Image preprocessing -----------------------------------------------------
# ResNet preprocessing for the CNN encoder; HF image processors for ViT/CLIP.
resnet_transform = T.Compose([
    T.Resize((256, 256)), T.CenterCrop(224), T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])

def preprocess_images(images, encoder_kind, image_processor):
    if encoder_kind == "cnn":
        return torch.stack([resnet_transform(im) for im in images], 0)
    return image_processor(list(images), return_tensors="pt").pixel_values

# ---- Encoder: image -> sequence of feature vectors (B, S, D_enc) -------------
class ImageEncoder(nn.Module):
    def __init__(self, kind, name):
        super().__init__()
        self.kind = kind
        if kind == "cnn":
            resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
            # Drop avgpool + fc -> keep the conv feature map (B, 2048, 7, 7).
            self.backbone = nn.Sequential(*list(resnet.children())[:-2])
            self.feat_dim = 2048
        elif kind == "clip":
            self.backbone = CLIPVisionModel.from_pretrained(name)
            self.feat_dim = self.backbone.config.hidden_size
        else:  # vit
            self.backbone = AutoModel.from_pretrained(name)
            self.feat_dim = self.backbone.config.hidden_size

    def forward(self, images):
        if self.kind == "cnn":
            f = self.backbone(images)                       # (B, 2048, 7, 7)
            B, C, H, W = f.shape
            return f.view(B, C, H * W).permute(0, 2, 1)     # (B, 49, 2048)
        out = self.backbone(pixel_values=images)
        return out.last_hidden_state                        # (B, T, 768)

# ---- Decoder A: word-level GRU (the earlier notebook's decoder) ---------------
class GRUDecoder(nn.Module):
    def __init__(self, feat_dim, vocab, embed_size, hidden_size, num_layers, dropout):
        super().__init__()
        self.vocab = vocab
        self.pad_id = vocab.word2idx["<pad>"]
        self.end_id = vocab.word2idx["<end>"]
        self.img_proj = nn.Linear(feat_dim, embed_size)
        self.bn = nn.BatchNorm1d(embed_size)
        self.embed = nn.Embedding(len(vocab), embed_size)
        self.dropout = nn.Dropout(dropout)
        self.rnn = nn.GRU(embed_size, hidden_size, num_layers, batch_first=True)
        self.linear = nn.Linear(hidden_size, len(vocab))
        self.criterion = nn.CrossEntropyLoss(ignore_index=self.pad_id)

    def _img_token(self, enc_seq):
        pooled = enc_seq.mean(dim=1)                  # (B, feat_dim)
        return self.bn(self.img_proj(pooled))         # (B, embed_size)

    def forward(self, enc_seq, captions):
        """captions: padded word ids (B, T). Returns scalar loss."""
        feat = self._img_token(enc_seq)
        emb = self.dropout(self.embed(captions[:, :-1]))
        inputs = torch.cat((feat.unsqueeze(1), emb), dim=1)   # (B, T, E)
        hiddens, _ = self.rnn(inputs)
        logits = self.linear(hiddens)                         # (B, T, V)
        ml = min(logits.size(1), captions.size(1))
        return self.criterion(
            logits[:, :ml, :].reshape(-1, logits.size(-1)),
            captions[:, :ml].reshape(-1))

    @torch.no_grad()
    def generate(self, enc_seq, max_len):
        feat = self._img_token(enc_seq)
        B = feat.size(0)
        inp = feat.unsqueeze(1)
        states = None
        done = torch.zeros(B, dtype=torch.bool, device=feat.device)
        seqs = [[] for _ in range(B)]
        for _ in range(max_len):
            hiddens, states = self.rnn(inp, states)
            pred = self.linear(hiddens.squeeze(1)).argmax(1)  # (B,)
            for i in range(B):
                if done[i]:
                    continue
                tid = int(pred[i].item())
                if tid == self.end_id:
                    done[i] = True
                else:
                    seqs[i].append(tid)
            if bool(done.all()):
                break
            inp = self.embed(pred).unsqueeze(1)
        return seqs

    def decode(self, ids):
        return self.vocab.decode(ids)

# ---- Decoder B: GPT-2 with cross-attention over the encoder sequence ----------
class GPT2Decoder(nn.Module):
    def __init__(self, feat_dim, tokenizer):
        super().__init__()
        cfg = AutoConfig.from_pretrained("gpt2")
        cfg.is_decoder = True
        cfg.add_cross_attention = True            # adds (random-init) cross-attn layers
        self.gpt2 = AutoModelForCausalLM.from_pretrained("gpt2", config=cfg)
        # Give GPT-2 a dedicated pad token so it doesn't collide with eos (both 50256).
        self.gpt2.resize_token_embeddings(len(tokenizer))
        self.enc_proj = nn.Linear(feat_dim, cfg.n_embd)
        self.tokenizer = tokenizer
        self.bos_id = tokenizer.bos_token_id
        self.eos_id = tokenizer.eos_token_id
        self.pad_id = tokenizer.pad_token_id

    def forward(self, enc_seq, token_ids):
        """token_ids: padded [bos]+caption+[eos] (B, T). Returns scalar loss."""
        enc_hidden = self.enc_proj(enc_seq)
        attn = (token_ids != self.pad_id).long()
        labels = token_ids.clone()
        labels[token_ids == self.pad_id] = -100   # ignore pad in the loss
        out = self.gpt2(input_ids=token_ids, attention_mask=attn,
                        encoder_hidden_states=enc_hidden, labels=labels)
        return out.loss

    @torch.no_grad()
    def generate(self, enc_seq, max_len):
        # GPT-2's generate() drops encoder_hidden_states, so decode manually with
        # KV-caching, feeding the encoder features at every step.
        enc_hidden = self.enc_proj(enc_seq)
        B = enc_hidden.size(0)
        dev = enc_hidden.device
        cur = torch.full((B, 1), self.bos_id, dtype=torch.long, device=dev)
        seqs = [[] for _ in range(B)]
        done = torch.zeros(B, dtype=torch.bool, device=dev)
        past = None
        for _ in range(max_len):
            out = self.gpt2(input_ids=cur, encoder_hidden_states=enc_hidden,
                            past_key_values=past, use_cache=True)
            past = out.past_key_values
            nxt = out.logits[:, -1, :].argmax(-1)             # (B,)
            for i in range(B):
                if done[i]:
                    continue
                tid = int(nxt[i].item())
                if tid == self.eos_id:
                    done[i] = True
                else:
                    seqs[i].append(tid)
            if bool(done.all()):
                break
            cur = nxt.unsqueeze(1)
        return seqs

    def decode(self, ids):
        return self.tokenizer.decode(ids, skip_special_tokens=True).strip()

# ---- The full model: encoder + decoder --------------------------------------
class CaptioningModel(nn.Module):
    def __init__(self, encoder, decoder, freeze_encoder=True):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.freeze_encoder = freeze_encoder

    def _encode(self, images):
        if self.freeze_encoder:
            with torch.no_grad():
                return self.encoder(images)
        return self.encoder(images)

    def forward(self, images, labels):
        return self.decoder(self._encode(images), labels)   # scalar loss

    @torch.no_grad()
    def generate(self, images, max_len):
        return self.decoder.generate(self._encode(images), max_len)  # list[list[int]]

    def decode(self, ids):
        return self.decoder.decode(ids)

# ---- Builder: returns the model + the bits collate/eval need -----------------
def build_model(encoder_spec, decoder_kind):
    kind, name = encoder_spec["kind"], encoder_spec["name"]
    encoder = ImageEncoder(kind, name)
    feat_dim = encoder.feat_dim

    if decoder_kind == "gru":
        decoder = GRUDecoder(feat_dim, rnn_vocab, GRU_EMBED_SIZE, GRU_HIDDEN_SIZE,
                             GRU_NUM_LAYERS, GRU_DROPOUT)
    elif decoder_kind == "gpt2":
        tok = AutoTokenizer.from_pretrained("gpt2")
        if tok.pad_token is None:
            tok.add_special_tokens({"pad_token": "<|pad|>"})  # distinct from eos
        decoder = GPT2Decoder(feat_dim, tok)
    else:
        raise ValueError(f"Unknown decoder kind: {decoder_kind}")

    model = CaptioningModel(encoder, decoder, freeze_encoder=FREEZE_ENCODER)
    if FREEZE_ENCODER:
        for p in model.encoder.parameters():
            p.requires_grad = False

    image_processor = None if kind == "cnn" else AutoImageProcessor.from_pretrained(name)
    return {"model": model, "encoder_kind": kind, "decoder_kind": decoder_kind,
            "image_processor": image_processor}

# ---- Collate: image preprocessing (encoder-specific) + label encoding (decoder-specific) ----
def make_collate(bundle):
    kind = bundle["encoder_kind"]
    dk = bundle["decoder_kind"]
    image_processor = bundle["image_processor"]
    tok = bundle["model"].decoder.tokenizer if dk == "gpt2" else None

    def collate(batch):
        images, captions, image_ids = zip(*batch)
        pixel_values = preprocess_images(images, kind, image_processor)
        if dk == "gru":
            seqs = [rnn_vocab.numericalize(c) for c in captions]
            maxlen = max(len(s) for s in seqs)
            labels = torch.zeros(len(seqs), maxlen, dtype=torch.long)  # 0 == <pad>
            for i, s in enumerate(seqs):
                labels[i, :len(s)] = torch.tensor(s, dtype=torch.long)
        else:  # gpt2: [bos] + tokens + [eos], padded with the dedicated pad id
            enc = tok(list(captions), add_special_tokens=False,
                      truncation=True, max_length=MAX_TEXT_LEN - 2)
            seqs = [[tok.bos_token_id] + ids + [tok.eos_token_id] for ids in enc["input_ids"]]
            maxlen = max(len(s) for s in seqs)
            labels = torch.full((len(seqs), maxlen), tok.pad_token_id, dtype=torch.long)
            for i, s in enumerate(seqs):
                labels[i, :len(s)] = torch.tensor(s, dtype=torch.long)
        return pixel_values, labels, torch.tensor(image_ids, dtype=torch.long)
    return collate

# ---- Training / evaluation (decoder-agnostic) -------------------------------
def train_one_epoch(model, loader, optimizer, device, max_batches=None):
    model.train()
    total_loss, n = 0.0, 0
    for batch_idx, (pixel_values, labels, _) in enumerate(loader):
        if max_batches is not None and batch_idx >= max_batches:
            break
        pixel_values = pixel_values.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        loss = model(pixel_values, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad], 1.0)
        optimizer.step()
        total_loss += float(loss.item()); n += 1
    return total_loss / max(n, 1)


def _tok(text):
    return word_tokenize(text.lower())


@torch.no_grad()
def evaluate_bleu(model, loader, device, num_batches=None):
    """BLEU-4 against all human references per image (greedy decoding)."""
    model.eval()
    references, hypotheses, seen = [], [], set()
    smoothing = SmoothingFunction().method1
    for batch_idx, (pixel_values, labels, image_ids) in enumerate(loader):
        if num_batches is not None and batch_idx >= num_batches:
            break
        pixel_values = pixel_values.to(device)
        gen_ids = model.generate(pixel_values, MAX_GEN_LEN)   # list[list[int]]
        for j in range(pixel_values.size(0)):
            img_id = int(image_ids[j].item())
            if img_id in seen:
                continue
            seen.add(img_id)
            hypotheses.append(_tok(model.decode(gen_ids[j])))
            references.append([_tok(c) for c in img_id_to_captions[img_id]])
    return 0.0 if not hypotheses else corpus_bleu(references, hypotheses, smoothing_function=smoothing)

## 7. Architecture search

We compare the cross-product of **encoders** (`cnn`, `vit`, `clip`) and **decoders** (`gru`, `gpt2`) — 6 combinations, including the original **CNN+GRU** baseline and novel mixes like **CNN+GPT-2** and **ViT+GRU**. Each combination trains on the subset for `EPOCHS` epochs and is scored by validation BLEU-4. The per-epoch best weights are kept (and saved) for each architecture, and the single best architecture overall drives the qualitative section.

Add or remove entries in `ENCODERS` / `DECODERS` to widen or narrow the comparison.

In [ ]:
# ---- Architecture grid -----------------------------------------------------
# Encoders are (kind, name) specs; decoders are kinds handled by build_model.
ENCODERS = [
    {"kind": "cnn",  "name": "resnet50"},                          # frozen ResNet-50
    {"kind": "vit",  "name": "google/vit-base-patch16-224-in21k"}, # ViT
    {"kind": "clip", "name": "openai/clip-vit-base-patch32"},      # CLIP vision tower
]
DECODERS = ["gru", "gpt2"]   # word-level GRU  vs  GPT-2 w/ cross-attention

ARCHITECTURES = [{"encoder": e, "decoder": d} for e in ENCODERS for d in DECODERS]
print(f"Comparing {len(ARCHITECTURES)} encoder/decoder combinations:")
for a in ARCHITECTURES:
    print(f"  - {a['encoder']['kind']} ({a['encoder']['name']})  +  {a['decoder']}")

In [ ]:
def short_name(arch):
    enc = arch["encoder"]["name"].split("/")[-1]
    return f"{arch['encoder']['kind']}_{enc}__{arch['decoder']}"


def run_architecture(arch):
    name = short_name(arch)
    decoder_kind = arch["decoder"]
    print("\n" + "=" * 80)
    print("Running:", name)
    print("=" * 80)

    bundle = build_model(arch["encoder"], decoder_kind)
    model = bundle["model"].to(device)

    collate = make_collate(bundle)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, collate_fn=collate,
                              pin_memory=torch.cuda.is_available())
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, collate_fn=collate,
                            pin_memory=torch.cuda.is_available())

    lr = LR_BY_DECODER[decoder_kind]
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad], lr=lr)

    save_dir = os.path.join(OUTPUT_DIR, name)
    losses, val_bleus = [], []
    best_bleu, best_epoch, best_state = -1.0, -1, None
    start = time.time()

    for epoch in range(EPOCHS):
        loss = train_one_epoch(model, train_loader, optimizer, device, max_batches=MAX_TRAIN_BATCHES)
        losses.append(loss)
        val_bleu = evaluate_bleu(model, val_loader, device, num_batches=EVAL_NUM_BATCHES)
        val_bleus.append(val_bleu)

        flag = ""
        if val_bleu > best_bleu:
            best_bleu, best_epoch = val_bleu, epoch
            best_state = copy.deepcopy(model.state_dict())
            flag = "  <-- best so far"
        print(f"Epoch {epoch+1}/{EPOCHS}: train loss = {loss:.4f}, val BLEU-4 = {val_bleu:.4f}{flag}")

    # Roll back to the best epoch and save that model's weights.
    if best_state is not None:
        model.load_state_dict(best_state)
    os.makedirs(save_dir, exist_ok=True)
    torch.save(model.state_dict(), os.path.join(save_dir, "model.pt"))
    if decoder_kind == "gpt2":
        model.decoder.tokenizer.save_pretrained(save_dir)
    print("Saved best weights to:", save_dir)

    # Loss / BLEU curve
    plt.figure(figsize=(6, 4))
    plt.plot(range(1, len(losses) + 1), losses, marker="o", label="train loss")
    plt.plot(range(1, len(val_bleus) + 1), val_bleus, marker="s", label="val BLEU-4")
    plt.title(name); plt.xlabel("Epoch"); plt.ylabel("loss / BLEU-4")
    plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
    curve_path = os.path.join(OUTPUT_DIR, f"{name}_curve.png")
    plt.savefig(curve_path, dpi=150); plt.show()

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    result = {
        "name": name,
        "encoder": f"{arch['encoder']['kind']} ({arch['encoder']['name']})",
        "decoder": decoder_kind,
        "best_epoch": best_epoch + 1,
        "final_train_loss": losses[-1],
        "bleu4": best_bleu,
        "learning_rate": lr,
        "train_time_seconds": round(time.time() - start, 2),
        "trainable_params": trainable,
        "save_dir": save_dir,
        "curve": curve_path,
    }
    return result, bundle


results, best_bundle, best_overall = [], None, -1

for arch in ARCHITECTURES:
    result, bundle = run_architecture(arch)
    results.append(result)

    if result["bleu4"] > best_overall:
        best_overall = result["bleu4"]
        best_bundle = {**bundle, "name": result["name"]}
    else:
        del bundle["model"]
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

results_df = pd.DataFrame(results).sort_values("bleu4", ascending=False).reset_index(drop=True)
results_csv = os.path.join(OUTPUT_DIR, "architecture_comparison.csv")
results_df.to_csv(results_csv, index=False)
print("\nSaved comparison table:", results_csv)
print("Best architecture:", best_bundle["name"], "| BLEU-4 =", round(best_overall, 4))
results_df

## 8. Save the best model to Google Drive (persistent storage)

`OUTPUT_DIR` is on the ephemeral Colab VM disk, so it is wiped when the runtime recycles. This copies the best architecture's saved model folder (weights + tokenizer + image processor) and the comparison CSV to Drive. On a local Jupyter it just reports the local path.

In [ ]:
import shutil

DRIVE_SAVE_DIR = "/content/drive/MyDrive/image_captioning_transformer"
best_name = best_bundle["name"]
best_dir = os.path.join(OUTPUT_DIR, best_name)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
    dest = os.path.join(DRIVE_SAVE_DIR, best_name)
    if os.path.exists(dest):
        shutil.rmtree(dest)
    shutil.copytree(best_dir, dest)   # model.pt (+ tokenizer for gpt2)
    print("Saved best model to Drive:", dest)
    if os.path.exists(results_csv):
        shutil.copy2(results_csv, os.path.join(DRIVE_SAVE_DIR, "architecture_comparison.csv"))
        print("Saved comparison CSV to Drive.")
except ImportError:
    print("Not on Colab — model already persists locally at:", os.path.abspath(best_dir))

## 9. Qualitative examples from the best architecture

In [ ]:
model = best_bundle["model"]
best_name = best_bundle["name"]
encoder_kind = best_bundle["encoder_kind"]
image_processor = best_bundle["image_processor"]
model.eval()
print("Best architecture:", best_name)

@torch.no_grad()
def caption_image(path):
    image = Image.open(path).convert("RGB")
    # Same encoder-specific preprocessing used during training.
    pixel_values = preprocess_images([image], encoder_kind, image_processor).to(device)
    gen_ids = model.generate(pixel_values, MAX_GEN_LEN)   # list[list[int]]
    return image, model.decode(gen_ids[0])

sample_eval_ids = random.sample(list(val_img_ids), min(6, len(val_img_ids)))
fig, axes = plt.subplots(2, 3, figsize=(16, 9)); axes = axes.flatten()
rows = []
for ax, img_id in zip(axes, sample_eval_ids):
    fname = img_id_to_filename[img_id]
    image, pred = caption_image(os.path.join(img_dir, fname))
    ax.imshow(image); ax.set_title(pred, fontsize=9, wrap=True); ax.axis("off")
    refs = img_id_to_captions[img_id]
    rows.append({"image_id": img_id, "file_name": fname, "generated_caption": pred,
                 "reference_caption_1": refs[0],
                 "reference_caption_2": refs[1] if len(refs) > 1 else ""})
plt.suptitle("Generated captions: " + best_name); plt.tight_layout()
qual_path = os.path.join(OUTPUT_DIR, "generated_caption_examples.png")
plt.savefig(qual_path, dpi=150, bbox_inches="tight"); plt.show()

qual_df = pd.DataFrame(rows)
qual_df.to_csv(os.path.join(OUTPUT_DIR, "qualitative_examples.csv"), index=False)
print("Saved examples:", qual_path)
qual_df

## 10. Report-ready notes

- **What we compared:** a 3×2 grid of **encoders** (CNN/ResNet-50, ViT, CLIP-ViT) × **decoders** (word-level GRU, GPT-2 with cross-attention) — 6 architectures scored by validation BLEU-4. This folds the original CNN+GRU model into the same search and adds cross-family mixes (e.g. CNN+GPT-2, ViT+GRU) that off-the-shelf `VisionEncoderDecoderModel` can't express.
- **Setup:** encoder frozen (feature extractor); the decoder (plus the GRU's image projection or GPT-2's `enc_proj` + cross-attention) is trained for 3 epochs on a subset. Greedy decoding is used for both the BLEU table and the qualitative examples, and the per-epoch best weights are rolled back before scoring/saving. LR is decoder-specific (GRU `1e-3`, GPT-2 `5e-5`).
- **Caveats for the writeup:** GPT-2's cross-attention layers are randomly initialized, so a few epochs on a small subset gives modest absolute BLEU — the comparison is *relative*. For stronger numbers: unfreeze the encoder (`FREEZE_ENCODER = False`), raise `EPOCHS` / `MAX_TRAIN_BATCHES`, or switch `COCO_SPLIT` to `train`. A pretrained captioning checkpoint (e.g. `nlpconnect/vit-gpt2-image-captioning`) would start far higher but wouldn't be a fair from-scratch architecture comparison.
- **Outputs:** `architecture_comparison.csv` (quantitative table), `*_curve.png` (training curves), `generated_caption_examples.png` + `qualitative_examples.csv` (qualitative).